# 10 · Improve model performance

## What you will build

A toolkit for making a model better on purpose rather than by luck. You will read learning curves to tell underfitting from overfitting, run experiments on the learning rate, the batch size and the initial weights, and type three tools yourself:

1. `l2_penalty` and `add_l2_grads`: L2 regularisation, also called weight decay;
2. `train_with_early_stopping`: training that keeps the best checkpoint and stops once validation stops improving;
3. `confusion_matrix` and `precision_recall_f1`: metrics that show *which* mistakes a classifier makes.

## Learning objectives

- Diagnose underfitting and overfitting from training and validation curves, and choose a fix for each.
- Choose a learning rate with a log-spaced sweep, and explain what the batch size trades off.
- Explain why initial weights must be random and suitably scaled, and recognise what bad initialisation looks like.
- Implement L2 regularisation and early stopping, and compute a confusion matrix, precision, recall and F1 from scratch.

## Dependencies

Tier 1: NumPy and Matplotlib only. All the data are generated spirals.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# From notebook 07: backpropagation.

def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy: (P - Y) / n."""
    return (probs - np.eye(probs.shape[1])[y]) / len(y)


def dense_backward(dZ, A_prev, W):
    """Gradients of a dense layer: (dW, db, dA_prev)."""
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T


def backward(network, cache, probs, y):
    """Every layer's gradients, last layer first, returned in layer order."""
    grads = [None] * len(network)
    dZ = softmax_cross_entropy_grad(probs, y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
        grads[i] = {"W": dW, "b": db}
    return grads


# From notebook 08: training with gradient descent.

def sgd_step(network, grads, lr):
    """A new network one gradient-descent step downhill: W - lr * dW, b - lr * db."""
    return [{"W": layer["W"] - lr * g["W"], "b": layer["b"] - lr * g["b"]} for layer, g in zip(network, grads)]


def minibatch_indices(n, batch_size, seed):
    """A shuffled split of range(n) into batches of batch_size (the last may be smaller)."""
    order = np.random.default_rng(seed).permutation(n)
    return [order[start:start + batch_size] for start in range(0, n, batch_size)]


def train_epoch(network, X, y, lr, batch_size, seed):
    """One pass over the data in shuffled mini-batches. Returns (network, mean batch loss)."""
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        network = sgd_step(network, backward(network, cache, probs, y[batch]), lr)
    return network, float(np.mean(losses))


def train(network, X, y, X_val, y_val, lr, epochs, batch_size, seed=0):
    """Train for some epochs; returns (network, history of losses and accuracies per epoch)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch(network, X, y, lr, batch_size, seed + epoch)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history

## 1 · A small, noisy problem

To study the problems that tuning fixes, it helps to provoke them on purpose. The spirals here have heavy noise (0.6 radians of random jitter, so the arms overlap) and only 25 training points per class. Three independent draws play three roles: training data, validation data for decisions, and test data that we keep for the very end.

Shapes: `X_train` is `(75, 2)`, while `X_val` and `X_test` are `(450, 2)`; every `y` is a vector of class indices 0, 1 and 2.

In [ ]:
NOISE = 0.6
X_train, y_train = make_spirals(25, 3, NOISE, np.random.default_rng(1))   # (75, 2): deliberately few
X_val, y_val = make_spirals(150, 3, NOISE, np.random.default_rng(2))      # (450, 2): for decisions
X_test, y_test = make_spirals(150, 3, NOISE, np.random.default_rng(3))    # (450, 2): used once, at the end
print(X_train.shape, X_val.shape, X_test.shape)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap="viridis", s=6, alpha=0.3)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=30, edgecolors="k", linewidths=0.5)
ax.set_title("training points (large) and validation points (small)")
plt.show()

Three helpers keep the experiments short. `train_fresh` initialises a network from a seed and trains it with your `train`, always validating on `X_val`. `plot_runs` draws the training and validation curves of several runs side by side, and `plot_regions` colours the plane by each network's predicted class.

In [ ]:
def train_fresh(sizes, X, y, lr=0.1, epochs=500, batch_size=10, seed=0):
    """He-initialise a network from seed and train it on (X, y), validating on X_val. Returns (network, history)."""
    return train(init_network(sizes, np.random.default_rng(seed)), X, y, X_val, y_val, lr, epochs, batch_size, seed)


def plot_runs(histories, titles, key="loss"):
    """One panel per run with its training and validation curves; key is "loss" or "acc"."""
    fig, axes = plt.subplots(1, len(histories), figsize=(3.8 * len(histories), 3), sharey=True, squeeze=False)
    for ax, history, title in zip(axes[0], histories, titles):
        ax.plot(history[f"train_{key}"], label="training")
        ax.plot(history[f"val_{key}"], label="validation")
        ax.set_title(title)
        ax.set_xlabel("epoch")
    axes[0][0].set_ylabel("cross-entropy" if key == "loss" else "accuracy")
    axes[0][0].legend()
    plt.tight_layout()
    plt.show()


def plot_regions(networks, titles):
    """Colour the plane by each network's predicted class, with the training points on top."""
    grid = np.linspace(-1.15, 1.15, 120)
    gx, gy = np.meshgrid(grid, grid)
    points = np.column_stack([gx.ravel(), gy.ravel()])
    fig, axes = plt.subplots(1, len(networks), figsize=(3.4 * len(networks), 3.4), squeeze=False)
    for ax, network, title in zip(axes[0], networks, titles):
        ax.contourf(gx, gy, predict(network, points).reshape(gx.shape), levels=[-0.5, 0.5, 1.5, 2.5],
                    cmap="viridis", alpha=0.3)
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=16, edgecolors="k", linewidths=0.4)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

## 2 · Underfitting and overfitting

Every model sits somewhere between two failures.

- **Underfitting**: the model is too simple, or trained too briefly, to capture the pattern. The training loss stays high and the validation loss is about as high: both sets are fitted equally badly.
- **Overfitting**: the model is flexible enough to memorise its training points, noise included. The training loss keeps falling towards 0, while the validation loss reaches a minimum and then rises, so the gap between the curves grows.

| What you see | Diagnosis | What to try |
|---|---|---|
| Training loss high, validation loss close to it | underfitting | a bigger network, more epochs, a better learning rate |
| Training loss near 0, validation loss much higher and rising | overfitting | more data, regularisation, early stopping, a smaller network |
| Both low and close together | a good fit | stop tuning, and evaluate the test set once |

### 🧪 Capacity: three network sizes

Train a tiny, a medium and a large network on the same 75 points for 500 epochs. Edit `hidden_layers` and run the cell again to try other sizes.

In [ ]:
hidden_layers = {"tiny [2]": [2], "medium [16]": [16], "large [64, 64]": [64, 64]}  # 🧪 try [8] or [128, 128]
capacity_runs = {}
for name, hidden in hidden_layers.items():
    network, history = train_fresh([2, *hidden, 3], X_train, y_train)
    capacity_runs[name] = (network, history)
    lowest = int(np.argmin(history["val_loss"]))
    print(f"{name:>15}: accuracy {history['train_acc'][-1]:.3f} training, {history['val_acc'][-1]:.3f} validation; "
          f"validation loss lowest {history['val_loss'][lowest]:.3f} at epoch {lowest}, final {history['val_loss'][-1]:.3f}")
plot_runs([history for _, history in capacity_runs.values()], list(capacity_runs))
plot_regions([network for network, _ in capacity_runs.values()], list(capacity_runs))

**What to notice.** The tiny network underfits: about 0.52 training and 0.47 validation accuracy, with both losses stuck near 1. The large one overfits: it gets about 99% of its training points right, and its validation loss falls to about 0.62 at epoch 160 and then climbs to about 0.85, even though its validation accuracy (about 0.85) is the best of the three. Rising validation loss with steady accuracy means the network has become confidently wrong on the points it misses. In the region plots, the large network carves small pockets around individual training points: it has learned the noise.

### 🧪 More data

The most reliable cure for overfitting is more data. Train the large network on 25, 100 and 300 points per class, with about 4,000 updates each so that only the amount of data changes.

In [ ]:
points_per_class = [25, 100, 300]  # 🧪 try 50 or 200
data_rows = []
for n in points_per_class:
    X_more, y_more = make_spirals(n, 3, NOISE, np.random.default_rng(10 + n))
    epochs = max(1, round(4000 / len(minibatch_indices(len(X_more), 10, 0))))
    _, history = train_fresh([2, 64, 64, 3], X_more, y_more, epochs=epochs)
    data_rows.append((n, history["train_acc"][-1], history["val_acc"][-1]))
    print(f"{n:3d} per class ({epochs:3d} epochs): accuracy {history['train_acc'][-1]:.3f} training, "
          f"{history['val_acc'][-1]:.3f} validation, gap {history['train_acc'][-1] - history['val_acc'][-1]:.3f}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot([row[0] for row in data_rows], [row[1] for row in data_rows], "o-", label="training")
ax.plot([row[0] for row in data_rows], [row[2] for row in data_rows], "s-", label="validation")
ax.set_xscale("log")
ax.set_xlabel("training points per class")
ax.set_ylabel("final accuracy")
ax.legend()
plt.show()

**What to notice.** With 25 points per class the gap between training and validation accuracy is about 0.16; with 100 it shrinks to about 0.06, and with 300 it almost disappears, while the validation accuracy rises from about 0.81 to about 0.89. A bigger dataset leaves the network less room to memorise. When you cannot get more data, the techniques in the rest of this notebook are the next best thing.

## 3 · Choosing the learning rate

For a bowl-shaped loss with curvature $a$, gradient descent is stable only when the learning rate satisfies $\eta < 2/a$. A network's loss has many curvatures at once, so in practice you search: too small and training crawls, too large and the loss jumps around or blows up. Try values a factor of 10 apart (a **log-spaced sweep**), keep the one with the lowest validation loss, then refine between its neighbours.

### 🧪 A learning-rate sweep

Train the large network for 100 epochs with four learning rates. Add values to the list and run the cell again.

In [ ]:
learning_rates = [0.003, 0.03, 0.3, 3.0]  # 🧪 add 0.1 or 1.0
lr_histories = {}
for lr in learning_rates:
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        _, lr_histories[lr] = train_fresh([2, 64, 64, 3], X_train, y_train, lr=lr, epochs=100)
    h = lr_histories[lr]
    print(f"lr {lr:<6}: training loss {h['train_loss'][0]:.3f} after epoch 1, {h['train_loss'][-1]:.3f} after 100; "
          f"lowest validation loss {min(h['val_loss']):.3f}")
best_lr = min(learning_rates, key=lambda lr: min(lr_histories[lr]["val_loss"]))
print("lowest validation loss with lr =", best_lr)

fig, ax = plt.subplots(figsize=(6, 3.4))
for lr in learning_rates:
    ax.plot(lr_histories[lr]["train_loss"], label=f"lr = {lr}")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.legend()
plt.show()

**What to notice.** At 0.003 the loss barely moves (from about 1.11 to 1.01). At 0.03 it falls steadily but slowly. At 0.3 it falls fast but jumps up and down, and it reaches the lowest validation loss, about 0.585. At 3.0 training blows up: the loss rises and the network ends up predicting a single class. The sweep says the right order of magnitude is 0.1 to 0.3. From here on we use 0.1, whose curves are smoother, so that the effects of the other changes are easier to see.

## 4 · Batch size

The batch size decides how many examples each update averages over. Small batches give many noisy updates per epoch; large batches give a few smooth ones. With $n$ training points and batch size $B$, an epoch makes $\lceil n/B \rceil$ updates. Each update also has a fixed cost (Python overhead, function calls), so on a CPU an epoch of tiny batches takes longer. Common choices are 32 to 256 on larger datasets; when you raise the batch size a lot, a larger learning rate is often needed to keep the same progress per epoch.

### 🧪 Four batch sizes

Train for 100 epochs with batch sizes from 1 (pure stochastic gradient descent) to 75 (the whole training set: full-batch gradient descent).

In [ ]:
import time

batch_sizes = [1, 5, 25, 75]  # 🧪 try 10 or 15; 75 is the whole training set
batch_histories = {}
for batch_size in batch_sizes:
    start = time.perf_counter()
    _, batch_histories[batch_size] = train_fresh([2, 64, 64, 3], X_train, y_train, epochs=100, batch_size=batch_size)
    ms_per_epoch = (time.perf_counter() - start) / 100 * 1000
    updates = len(minibatch_indices(len(X_train), batch_size, 0))
    print(f"batch size {batch_size:2d}: {updates:2d} updates per epoch, {ms_per_epoch:5.2f} ms per epoch, "
          f"lowest validation loss {min(batch_histories[batch_size]['val_loss']):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for batch_size in batch_sizes:
    val_loss = batch_histories[batch_size]["val_loss"]
    updates = len(minibatch_indices(len(X_train), batch_size, 0))
    axes[0].plot(val_loss, label=f"batch {batch_size}")
    axes[1].plot(np.arange(1, len(val_loss) + 1) * updates, val_loss, label=f"batch {batch_size}")
axes[0].set_xlabel("epoch")
axes[1].set_xlabel("number of updates")
axes[1].set_xscale("log")
axes[0].set_ylabel("validation loss")
axes[0].legend()
plt.tight_layout()
plt.show()

**What to notice.** Per epoch, the small batches make the most progress, because they take 75 or 15 steps where the full batch takes one; the full-batch run is still far from a good fit after 100 epochs. Per update (right panel), larger batches make steadier progress, because each step averages more examples. An epoch with batch size 1 takes roughly ten times as long as one with the full batch: most of the time goes on the fixed cost of each update, not on arithmetic. The best batch size balances noise, speed and memory, and the learning rate has to be tuned together with it.

## 5 · Weight initialisation

Where the weights start matters as much as how they move. Here are four choices for a deeper network, `[2, 32, 32, 32, 3]`:

- **All zeros.** Every hidden pre-activation is then $Z = 0$, every ReLU outputs 0 and `relu_grad(0)` is 0. The gradient of every hidden weight is exactly zero, and so is the gradient of the output weights, because the activations they multiply are zero. Only the output biases learn, so the network can at best learn how common each class is, and the loss stays at $\ln 3 \approx 1.0986$. The network is dead from the start, not merely symmetric.
- **All equal but not zero** (0.1 everywhere). The units in a layer start identical, receive identical gradients and therefore stay identical forever, so 32 units act like 1. This is the **symmetry** problem.
- **Tiny random** (standard deviation 0.01). The symmetry is broken, but each layer scales the signal by roughly $0.01\sqrt{n_\text{in}/2}$, about 0.04 for 32 inputs. After three hidden layers the signal is about $10^{-5}$ of its starting size, the logits are almost equal, and the gradients reaching the early layers are tiny.
- **He initialisation**, drawing $W \sim \mathcal{N}(0,\, 2/n_\text{in})$, keeps the spread of the activations roughly constant through ReLU layers, so signals and gradients keep a useful size.

`init_with` builds layers like `init_network`, but with the weights drawn by a chosen scheme. The demo measures the spread of each hidden layer's output on the validation points, straight after initialisation.

In [ ]:
def init_with(sizes, rng, scheme):
    """Layers like init_network's, with weights drawn by scheme: "zeros", "constant", "tiny" or "he"."""
    layers = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        if scheme == "zeros":
            W = np.zeros((n_in, n_out))
        elif scheme == "constant":
            W = np.full((n_in, n_out), 0.1)
        elif scheme == "tiny":
            W = rng.normal(0.0, 0.01, (n_in, n_out))
        else:
            W = rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out))
        layers.append({"W": W, "b": np.zeros(n_out)})
    return layers


DEEP = [2, 32, 32, 32, 3]
for scheme in ("tiny", "he"):
    _, cache = forward(init_with(DEEP, np.random.default_rng(0), scheme), X_val)
    spreads = [f"{relu(Z).std():.1e}" for _, Z in cache[:-1]]
    print(f"{scheme:>4}: standard deviation of each hidden layer's output: {spreads}")

### 🧪 Four initialisations

Train each initialisation for 150 epochs and count the **distinct units** in the second hidden layer: the number of different columns in its weight matrix. Change a scheme inside `init_with` (for example the constant 0.1) and run the cell again.

In [ ]:
schemes = ["zeros", "constant", "tiny", "he"]  # 🧪 edit init_with (for example the constant) and rerun
for scheme in schemes:
    network, history = train(init_with(DEEP, np.random.default_rng(0), scheme), X_train, y_train, X_val, y_val,
                             0.1, 150, 10)
    distinct = np.unique(network[1]["W"].round(6), axis=1).shape[1]
    print(f"{scheme:>8}: training loss {history['train_loss'][0]:.4f} -> {history['train_loss'][-1]:.4f}, "
          f"validation accuracy {history['val_acc'][-1]:.3f}, distinct units in layer 2: {distinct}")

**What to notice.** In the demo, the tiny initialisation's hidden outputs shrink from about $3 \times 10^{-3}$ to $4 \times 10^{-6}$, while He's stay near 0.3. In training, zeros and tiny weights both stay at a loss of 1.0987 ($\ln 3$) for all 150 epochs and predict a single class. The constant start learns a little (the loss falls to about 1.03), but its 32 units never become different: there is only 1 distinct unit. He initialisation trains normally, to about 0.85 validation accuracy, with all 32 units distinct.

## 6 · L2 regularisation (weight decay)

An overfitting network often uses large weights to bend its boundary around single points. **L2 regularisation** adds a penalty on the size of the weights to the loss:

$$L_\text{total} = L_\text{CE} + \frac{\lambda}{2} \sum_{\ell} \sum_{i,j} \big(W^{(\ell)}_{ij}\big)^2 ,$$

where $\lambda \ge 0$ sets its strength and the sum runs over the weight matrices of every layer. The biases are not penalised: they shift the boundary without making it more wiggly. The penalty's gradient is simply $\lambda W$, so each update becomes

$$W \leftarrow W - \eta\,\big(\nabla_W L_\text{CE} + \lambda W\big) = (1 - \eta\lambda)\,W - \eta\,\nabla_W L_\text{CE} ,$$

which shrinks every weight by the factor $1 - \eta\lambda$ before the usual step: hence the other name, **weight decay**. Smaller weights give a smoother function that is less able to fit noise.

Shapes: the penalty is one number; the gradients keep the structure of the network, a list of `{"W": (n_in, n_out), "b": (n_out,)}` dicts.

The demo compares the squared size of the large network's weights before and after its 500 epochs of overfitting.

In [ ]:
large_network = capacity_runs["large [64, 64]"][0]
fresh = init_network([2, 64, 64, 3], np.random.default_rng(0))
print("sum of squared weights at the start:", round(sum(float(np.sum(layer["W"] ** 2)) for layer in fresh), 1))
print("after 500 epochs of overfitting:   ", round(sum(float(np.sum(layer["W"] ** 2)) for layer in large_network), 1))

### ✍️ Your turn: `l2_penalty`

Write `l2_penalty(network, lam)`, returning $\frac{\lambda}{2}\sum_\ell \sum_{i,j} (W^{(\ell)}_{ij})^2$ as a Python `float`. Use only each layer's `"W"`; ignore the biases.

Hint: `np.sum(layer["W"] ** 2)` for each layer, summed over the layers.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def l2_penalty_reference(network, lam):
    return 0.5 * lam * float(sum(np.sum(layer["W"] ** 2) for layer in network))

In [ ]:
def l2_penalty(network, lam):
    """(lam / 2) times the sum of every squared weight (biases excluded), as a float."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
net_check = init_network([2, 3, 2], np.random.default_rng(0))
l2_penalty = practice("l2_penalty", l2_penalty, l2_penalty_reference, cases=[
    (net_check, 0.1),
    (net_check, 0.0),                                                    # edge case: no regularisation
    ([{"W": np.array([[1.0, -2.0]]), "b": np.array([100.0, 100.0])}], 1.0),  # biases must be ignored
])

**Expected behaviour.** The demo shows the weights' squared size nearly doubling during overfitting, from about 250 to about 485. For the third check, the penalty is $\frac{1}{2}(1^2 + (-2)^2) = 2.5$, whatever the biases are.

The penalty's gradient is added to the cross-entropy gradient of every weight matrix, $\nabla_W L_\text{total} = \nabla_W L_\text{CE} + \lambda W$, while the bias gradients stay as they are. Return new arrays rather than changing `grads` in place: a function that quietly changes its arguments makes bugs that are very hard to find, and `practice` would see the change too.

In [ ]:
W_demo = np.array([[1.0, -2.0]])
print("lam * W for lam = 0.1:", 0.1 * W_demo)

### ✍️ Your turn: `add_l2_grads`

Write `add_l2_grads(grads, network, lam)`, returning a **new** list with one dict per layer: `{"W": grads[i]["W"] + lam * network[i]["W"], "b": grads[i]["b"]}`. Do not modify `grads`.

Hint: a list comprehension over `zip(grads, network)`, like your `sgd_step`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def add_l2_grads_reference(grads, network, lam):
    return [{"W": grad["W"] + lam * layer["W"], "b": grad["b"]} for grad, layer in zip(grads, network)]

In [ ]:
def add_l2_grads(grads, network, lam):
    """New gradients with lam * W added to every weight gradient (biases unchanged)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
grads_check = [{"W": np.ones_like(layer["W"]), "b": np.full_like(layer["b"], 2.0)} for layer in net_check]
add_l2_grads = practice("add_l2_grads", add_l2_grads, add_l2_grads_reference, cases=[
    (grads_check, net_check, 0.1),
    (grads_check, net_check, 0.0),    # edge case: lam = 0 leaves the gradients unchanged
])

**Expected behaviour.** Each weight gradient grows by `lam * W` and each bias gradient stays at 2. If the check reports a difference of exactly `lam * W`, your function probably changed `grads` in place, so the reference saw an already-modified input.

`train_epoch_l2` is your `train_epoch` with one extra line: it passes the gradients through `add_l2_grads` before the update. `train_regularised` is your `train` built on it. With `lam = 0` both behave exactly like the originals.

In [ ]:
def train_epoch_l2(network, X, y, lr, batch_size, seed, lam):
    """One shuffled pass with L2-regularised updates. Returns (network, mean batch cross-entropy)."""
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        grads = add_l2_grads(backward(network, cache, probs, y[batch]), network, lam)
        network = sgd_step(network, grads, lr)
    return network, float(np.mean(losses))


def train_regularised(network, X, y, X_val, y_val, lr, epochs, batch_size, lam, seed=0):
    """Like train, with L2 regularisation. The recorded losses are plain cross-entropy, without the penalty."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch_l2(network, X, y, lr, batch_size, seed + epoch, lam)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history

### 🧪 How much regularisation?

Train the large network with three penalty strengths for 500 epochs. The recorded losses leave out the penalty, so that runs with different $\lambda$ can be compared on the same scale.

In [ ]:
lams = [0.0, 0.003, 0.03]  # 🧪 try 0.001 or 0.01
l2_runs = {}
for lam in lams:
    network, history = train_regularised(init_network([2, 64, 64, 3], np.random.default_rng(0)),
                                         X_train, y_train, X_val, y_val, 0.1, 500, 10, lam)
    l2_runs[lam] = (network, history)
    print(f"lam {lam:<6}: accuracy {history['train_acc'][-1]:.3f} training, {history['val_acc'][-1]:.3f} validation; "
          f"final validation loss {history['val_loss'][-1]:.3f}; penalty {l2_penalty(network, lam):.3f}")
plot_runs([history for _, history in l2_runs.values()], [f"lam = {lam}" for lam in lams])
plot_regions([network for network, _ in l2_runs.values()], [f"lam = {lam}" for lam in lams])

**What to notice.** Without regularisation the final validation loss is about 0.85. With $\lambda = 0.003$ it is about 0.51, and the regions become smoother, with fewer pockets around single points. With $\lambda = 0.03$ the penalty wins: the weights are squeezed so hard that the network barely learns (accuracy about 0.40, validation loss about 1.08). $\lambda$ is a hyperparameter like the learning rate, chosen on validation data.

## 7 · Early stopping with a best checkpoint

The validation loss usually falls, bottoms out and then rises as the network starts to memorise. **Early stopping** turns that into a rule: keep training while the validation loss improves; once it has not improved for `patience` epochs in a row, stop, and return the weights from the best epoch. Those saved weights are a **checkpoint**.

A checkpoint must be a real copy. Your `sgd_step` builds a new network at every step, so a name that points at an old network keeps the old numbers. Frameworks such as PyTorch update their arrays **in place** instead (`param -= lr * grad`), and then a "checkpoint" that is only a second name for the same arrays silently changes as training continues. The demo shows the trap and `copy_network`, which avoids it whichever kind of update you use.

In [ ]:
def copy_network(network):
    """New arrays with the same numbers: a checkpoint that later updates cannot change."""
    return [{"W": layer["W"].copy(), "b": layer["b"].copy()} for layer in network]


network = init_network([2, 4, 3], np.random.default_rng(0))
same_arrays = network                  # another name for the same arrays
checkpoint = copy_network(network)     # a real copy
network[0]["W"] -= 1.0                 # an in-place update, as frameworks make it
print("the second name changed too:", np.array_equal(same_arrays[0]["W"], network[0]["W"]))
print("the copy kept the old values:", not np.array_equal(checkpoint[0]["W"], network[0]["W"]))

### ✍️ Your turn: `train_with_early_stopping`

Write `train_with_early_stopping(network, X, y, X_val, y_val, lr, epochs, batch_size, patience, lam=0.0, seed=0)`, returning `(best_network, history, best_epoch)`.

1. Start with `history` holding four empty lists (`"train_loss"`, `"val_loss"`, `"train_acc"`, `"val_acc"`), `best_network = network`, `best_epoch = 0`, `best_loss = np.inf` and `waited = 0`.
2. For `epoch` in `range(epochs)`: run `network, _ = train_epoch_l2(network, X, y, lr, batch_size, seed + epoch, lam)`, then append the four metrics exactly as `train` does (cross-entropy and `accuracy` on the full training and validation sets).
3. If the new validation loss is **strictly lower** than `best_loss`, save `best_network = copy_network(network)`, set `best_epoch = epoch`, `best_loss` to that loss and `waited = 0`. Otherwise add 1 to `waited`, and `break` once `waited >= patience`.
4. Return `(best_network, history, best_epoch)`.

Hint: steps 2 and 3 are about ten lines inside one `for` loop.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def train_with_early_stopping_reference(network, X, y, X_val, y_val, lr, epochs, batch_size, patience, lam=0.0, seed=0):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_network, best_epoch, best_loss, waited = network, 0, np.inf, 0
    for epoch in range(epochs):
        network, _ = train_epoch_l2(network, X, y, lr, batch_size, seed + epoch, lam)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
        if history["val_loss"][-1] < best_loss:
            best_network, best_epoch, best_loss, waited = copy_network(network), epoch, history["val_loss"][-1], 0
        else:
            waited += 1
            if waited >= patience:
                break
    return best_network, history, best_epoch

In [ ]:
def train_with_early_stopping(network, X, y, X_val, y_val, lr, epochs, batch_size, patience, lam=0.0, seed=0):
    """Train until validation loss stops improving; return (best_network, history, best_epoch)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_network, best_epoch, best_loss, waited = network, 0, np.inf, 0
    # ✍️ Your code here (about 12 lines)
    raise NotImplementedError

In [ ]:
X_small, y_small = X_train[::3], y_train[::3]          # 25 points: overfits within a few epochs
X_small_val, y_small_val = X_val[::9], y_val[::9]     # 50 validation points
start_check = init_network([2, 16, 3], np.random.default_rng(4))
train_with_early_stopping = practice("train_with_early_stopping", train_with_early_stopping,
                                     train_with_early_stopping_reference, cases=[
    (start_check, X_small, y_small, X_small_val, y_small_val, 0.5, 60, 5, 3),          # stops early
    (start_check, X_small, y_small, X_small_val, y_small_val, 0.5, 6, 5, 100, 0.01),   # never stops early, with L2
])

**Expected behaviour.** In the first check case the validation loss stops improving after a few epochs, so the run stops long before 60 epochs and returns a network from before its last epoch. That is where a missing copy, or a wrong `break`, would show up as ❌. In the second case `patience` is larger than the number of epochs, so all 6 epochs run.

Now apply early stopping to the large, unregularised network, with a patience of 50 epochs and at most 500 epochs.

In [ ]:
stopped_network, stopped_history, stopped_best = train_with_early_stopping(
    init_network([2, 64, 64, 3], np.random.default_rng(0)), X_train, y_train, X_val, y_val, 0.1, 500, 10, patience=50)
full_history = capacity_runs["large [64, 64]"][1]
print(f"stopped after {len(stopped_history['val_loss'])} epochs; best epoch {stopped_best}")
print(f"validation loss of the returned network: {cross_entropy(forward(stopped_network, X_val)[0], y_val):.3f}")
print(f"validation loss after all 500 epochs:    {full_history['val_loss'][-1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(full_history["val_loss"], color="lightgrey", label="500 epochs, no stopping")
ax.plot(stopped_history["val_loss"], label="with early stopping")
ax.axvline(stopped_best, color="grey", linestyle=":", label="kept epoch")
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.legend()
plt.show()

**What to notice.** The run stops after about 213 epochs instead of 500 and returns the network from epoch 162, whose validation loss is about 0.62 rather than the 0.85 reached at the end of the full run. Early stopping needs no change to the model, and it saves training time. Its cost is one more hyperparameter, the patience: too small and a noisy curve stops training too soon; too large and it wastes epochs.

## 8 · Beyond accuracy: the confusion matrix, precision, recall and F1

Accuracy counts mistakes but says nothing about *which* mistakes. A **confusion matrix** $C$ for $K$ classes is a $K \times K$ table of counts: $C_{ij}$ is the number of examples whose true class is $i$ and whose predicted class is $j$. Rows are the truth and columns the prediction, so correct answers sit on the diagonal, and the matrix sums to the number of examples.

The demo shows why this matters. With 95 negatives and 5 positives, a lazy classifier that always answers 0 scores 95% accuracy while never finding a single positive.

Shapes: `y_true` and `y_pred` are `(n,)` integer arrays; the matrix is `(n_classes, n_classes)` of integers.

In [ ]:
y_demo = np.array([0] * 95 + [1] * 5)
lazy = np.zeros(100, dtype=int)
print("accuracy of always answering 0:", np.mean(lazy == y_demo))

### ✍️ Your turn: `confusion_matrix`

Write `confusion_matrix(y_true, y_pred, n_classes)`, returning an integer array of shape `(n_classes, n_classes)` where entry `[i, j]` counts the examples with true class `i` and predicted class `j`. A class that never occurs gives a row (or column) of zeros.

Hints: start from `np.zeros((n_classes, n_classes), dtype=int)`, then add 1 at `[t, p]` for every pair in `zip(y_true, y_pred)`. (`np.add.at` does the same without a loop.)

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def confusion_matrix_reference(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes):
    """Counts: entry [i, j] is the number of examples of true class i predicted as class j."""
    # ✍️ Your code here (about 4 lines)
    raise NotImplementedError

In [ ]:
confusion_matrix = practice("confusion_matrix", confusion_matrix, confusion_matrix_reference, cases=[
    (np.array([0, 0, 1, 2, 2, 2]), np.array([0, 1, 1, 2, 0, 2]), 3),
    (y_demo, lazy, 2),
    (np.array([0, 1, 0]), np.array([0, 1, 1]), 4),    # edge case: classes 2 and 3 never occur
])

**Expected behaviour.** The first case gives `[[1, 1, 0], [0, 1, 0], [1, 0, 2]]`: one class-0 example was called 1, and one class-2 example was called 0. The lazy classifier gives `[[95, 0], [5, 0]]`, and the last case has two all-zero rows and columns.

Three numbers per class $k$ come straight out of the matrix:

$$\text{precision}_k = \frac{C_{kk}}{\sum_i C_{ik}}, \qquad \text{recall}_k = \frac{C_{kk}}{\sum_j C_{kj}}, \qquad F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}} .$$

**Precision** looks down column $k$: of the examples *predicted* $k$, what fraction really are $k$? **Recall** looks along row $k$: of the examples that really are $k$, what fraction did the model find? **F1** is their harmonic mean, which is high only when both are. When a denominator is 0 (a class never predicted, or never present), we define the result as 0. The **macro average** is the plain mean over the classes, so every class counts equally, however rare it is.

In [ ]:
cm_demo = np.array([[95, 0], [5, 0]])
print("column sums (examples predicted as each class):", cm_demo.sum(axis=0))
print("row sums (examples truly in each class):        ", cm_demo.sum(axis=1))

### ✍️ Your turn: `precision_recall_f1`

Write `precision_recall_f1(cm)`, returning a tuple `(precision, recall, f1)` of float arrays, each with shape `(n_classes,)`, with 0 wherever a denominator is 0.

Hints: `tp = np.diag(cm)`; the predicted counts are `cm.sum(axis=0)` and the true counts `cm.sum(axis=1)`. `np.divide(a, b, out=np.zeros(len(a)), where=b > 0)` divides only where `b > 0` and leaves zeros elsewhere.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def precision_recall_f1_reference(cm):
    cm = np.asarray(cm, dtype=float)
    tp = np.diag(cm)
    predicted, actual = cm.sum(axis=0), cm.sum(axis=1)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, actual, out=np.zeros_like(tp), where=actual > 0)
    total = precision + recall
    f1 = np.divide(2 * precision * recall, total, out=np.zeros_like(tp), where=total > 0)
    return precision, recall, f1

In [ ]:
def precision_recall_f1(cm):
    """Per-class precision, recall and F1 from a confusion matrix; 0 where undefined."""
    # ✍️ Your code here (about 7 lines)
    raise NotImplementedError

In [ ]:
precision_recall_f1 = practice("precision_recall_f1", precision_recall_f1, precision_recall_f1_reference, cases=[
    (np.array([[1, 1, 0], [0, 1, 0], [1, 0, 2]]),),
    (cm_demo,),                                   # edge case: class 1 is never predicted
    (np.diag([3, 4, 5]),),
])
p_demo, r_demo, f_demo = precision_recall_f1(cm_demo)
print("lazy classifier: precision", p_demo.round(3), "recall", r_demo.round(3), "F1", f_demo.round(3),
      "macro F1", round(float(f_demo.mean()), 3))

**Expected behaviour.** For the lazy classifier, class 0 has precision 0.95, recall 1 and F1 about 0.974, while class 1 has 0 for all three, so the macro F1 is about 0.487: a far more honest summary than 95% accuracy. A perfect diagonal matrix gives 1 everywhere.

## 9 · Put it together

Combine what the experiments showed: the large network (enough capacity), a learning rate of 0.1, L2 regularisation with $\lambda = 0.003$ and early stopping with a patience of 50 epochs. Compare it on **validation** data with the plain large network from section 2; only then evaluate the chosen model on the test set, once.

In [ ]:
improved_network, improved_history, improved_best = train_with_early_stopping(
    init_network([2, 64, 64, 3], np.random.default_rng(0)), X_train, y_train, X_val, y_val,
    lr=0.1, epochs=600, batch_size=10, patience=50, lam=0.003)
plain_network, plain_history = capacity_runs["large [64, 64]"]
improved_val_loss = cross_entropy(forward(improved_network, X_val)[0], y_val)
print(f"validation loss: plain {plain_history['val_loss'][-1]:.3f}, improved {improved_val_loss:.3f} "
      f"(kept epoch {improved_best} of {len(improved_history['val_loss'])})")
print(f"validation accuracy: plain {plain_history['val_acc'][-1]:.3f}, improved {accuracy(improved_network, X_val, y_val):.3f}")

# Every choice is made. Now, once: the test set.
test_pred = predict(improved_network, X_test)
cm_test = confusion_matrix(y_test, test_pred, 3)
precision, recall, f1 = precision_recall_f1(cm_test)
test_accuracy = float(np.mean(test_pred == y_test))
print("\ntest confusion matrix (rows: true class, columns: predicted class):")
print(cm_test)
for k in range(3):
    print(f"class {k}: precision {precision[k]:.3f}, recall {recall[k]:.3f}, F1 {f1[k]:.3f}")
print(f"test accuracy {test_accuracy:.3f}, macro F1 {f1.mean():.3f}")

In [ ]:
# The L2 gradient must be the derivative of the L2 penalty: check one network with central differences.
check_net = init_network([2, 5, 3], np.random.default_rng(9))
zero_grads = [{"W": np.zeros_like(layer["W"]), "b": np.zeros_like(layer["b"])} for layer in check_net]
analytic = add_l2_grads(zero_grads, check_net, 0.1)[0]["W"]
numeric = numerical_gradient(lambda W: l2_penalty(check_net, 0.1), check_net[0]["W"])
assert relative_error(analytic, numeric) < 1e-6, "add_l2_grads should be the gradient of l2_penalty"

# Zero initialisation never breaks the symmetry: the loss stays at ln 3 and every hidden unit stays the same.
dead_network, dead_history = train(init_with(DEEP, np.random.default_rng(0), "zeros"), X_train, y_train,
                                   X_val, y_val, 0.1, 20, 10)
assert abs(dead_history["train_loss"][-1] - np.log(3)) < 1e-3, "an all-zero network should not learn"
assert np.unique(dead_network[1]["W"].round(6), axis=1).shape[1] == 1, "zero init keeps every unit identical"

# Regularisation and early stopping improved the overfitting network, judged on validation data.
assert improved_best < len(improved_history["val_loss"]) - 1, "early stopping should keep an earlier epoch"
assert np.isclose(improved_val_loss, min(improved_history["val_loss"])), "the best checkpoint must be returned"
assert improved_val_loss < plain_history["val_loss"][-1], "the improved model should have a lower validation loss"

# The test evaluation, once, with floors well below the measured values (about 0.87).
assert cm_test.sum() == len(y_test) and np.isclose(np.trace(cm_test) / cm_test.sum(), test_accuracy)
assert test_accuracy >= 0.75, f"expected at least 0.75 test accuracy, got {test_accuracy:.3f}"
assert f1.mean() >= 0.75, f"expected a macro F1 of at least 0.75, got {f1.mean():.3f}"
print("✅ L2 gradient checked, zero init diagnosed, overfitting reduced, test set evaluated once.")

**Expected behaviour.** The improved model keeps epoch 361 of 412, with a validation loss of about 0.50 against 0.85 for the plain network, and a similar validation accuracy (about 0.84). On the test set it reaches about 0.87 accuracy and a macro F1 of about 0.87, with every class between about 0.85 and 0.91 F1. The asserts confirm the L2 gradient numerically, the zero-initialisation failure and the improvement. Notice what did *not* help much: accuracy. Most of the gain is in the loss, because regularisation and early stopping mainly stop the network from being confidently wrong.

## Exercises

### Exercise 1 · ✍️ Balanced accuracy

Write `balanced_accuracy(cm)`: the mean of the per-class recalls, taken over the classes that occur (rows of `cm` with a positive sum). Return a Python float. For the lazy classifier it should give 0.5, the score of a coin toss, rather than 0.95.

In [ ]:
def balanced_accuracy_reference(cm):
    cm = np.asarray(cm, dtype=float)
    actual = cm.sum(axis=1)
    present = actual > 0
    return float(np.mean(np.diag(cm)[present] / actual[present]))

In [ ]:
def balanced_accuracy(cm):
    """Mean recall over the classes that occur in the confusion matrix."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
balanced_accuracy = practice("balanced_accuracy", balanced_accuracy, balanced_accuracy_reference, cases=[
    (cm_demo,), (np.diag([3, 4, 5]),), (np.array([[2, 1, 0], [0, 0, 0], [1, 0, 3]]),),
])
print("balanced accuracy on the test set:", round(balanced_accuracy(cm_test), 3))

### Exercise 2 · ✍️ A learning-rate schedule

A common trick is to lower the learning rate as training goes on: big steps early, small careful steps late. Write `step_decay(lr0, epoch, drop=0.5, every=100)`, returning `lr0 * drop ** (epoch // every)`. Then copy `train_regularised` into a new cell, use `step_decay(0.3, epoch)` as the learning rate of each epoch, and compare the validation curve with the constant rate of 0.3 from section 3.

In [ ]:
def step_decay_reference(lr0, epoch, drop=0.5, every=100):
    return lr0 * drop ** (epoch // every)

In [ ]:
def step_decay(lr0, epoch, drop=0.5, every=100):
    """The learning rate for an epoch: lr0, halved (by default) every 100 epochs."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
step_decay = practice("step_decay", step_decay, step_decay_reference, cases=[
    (0.3, 0), (0.3, 99), (0.3, 100), (0.3, 250), (1.0, 30, 0.1, 10),
])

### Exercise 3 · Open question: dropout

**Dropout** zeroes a random fraction $p$ of the hidden activations at every training step and scales the survivors by $1/(1-p)$, so that the expected activation is unchanged; at evaluation time nothing is dropped. What does the forward pass have to remember so that the backward pass sends gradients only through the units that were kept? Where would the random mask's seed come from, so that a run stays reproducible?

### Exercise 4 · Open question: where choices are made

In this notebook the learning rate, the batch size, $\lambda$ and the patience were all chosen by looking at validation results. Suppose you had picked $\lambda$ by its test accuracy instead. Why would the reported test accuracy then be too optimistic, and what would you need to obtain an honest number again?

## Common errors and debugging

- **The validation loss rises while the accuracy stays flat.** That is overfitting showing up as overconfidence: use early stopping or regularisation, even though the accuracy alone looks fine.
- **Penalising the biases.** Only the weights are decayed; `add_l2_grads` leaves the bias gradients alone.
- **Comparing losses that include the penalty.** A larger $\lambda$ always adds a larger penalty, so compare plain cross-entropy across runs, as `train_regularised` records it.
- **A checkpoint that keeps changing.** With in-place updates, `best = network` is just a second name; use `copy_network`.
- **Early stopping that stops at once.** The patience is too small for a noisy validation curve; increase it.
- **Precision and recall swapped.** Rows of the confusion matrix are the truth and columns the prediction: precision divides by a column sum, recall by a row sum.
- **`nan` or a loss stuck at $\ln 3$ from the first epoch.** Check the learning rate (too large) and the initialisation (zeros, or weights far too small).

## Summary

Improving a model starts with a diagnosis. Learning curves separate underfitting (both losses high) from overfitting (training loss low, validation loss rising). Capacity, more data, the learning rate, the batch size and the initialisation each change the curves in recognisable ways. L2 regularisation shrinks the weights at every step, and early stopping keeps the best checkpoint; together they cut the overfitting network's validation loss from about 0.85 to 0.50. A confusion matrix, with precision, recall and F1 per class, then shows which mistakes remain.

## Next

**11 · NumPy versus PyTorch**: rebuild your model with scikit-learn and PyTorch, compare accuracy, speed and code size on the same data, and build the layer classes that frameworks are made of.